In [0]:
customers_data = [
(101,'Rahul',28,'Male','Bangalore','Savings',75000,'Active',720,'2022-01-15',201),
(102,'Priya',34,'Female','Hyderabad','Current',125000,'Active',760,'2021-06-20',202),
(103,'Arjun',41,'Male','Chennai','Savings',54000,'Active',680,'2023-03-10',203),
(104,'Sneha',29,'Female','Mumbai','Savings',92000,'Active',735,'2022-08-05',204),
(105,'Vikram',46,'Male','Delhi','Current',185000,'Active',790,'2020-11-12',205),
(106,'Anjali',32,'Female','Bangalore','Savings',68000,'Active',710,'2023-01-18',201),
(107,'Kiran',39,'Male','Pune','Savings',45000,'Inactive',650,'2021-09-25',206),
(108,'Meena',27,'Female','Kochi','Savings',83000,'Active',725,'2022-12-01',207),
(109,'Suresh',52,'Male','Hyderabad','Current',210000,'Active',805,'2019-05-14',202),
(110,'Divya',31,'Female','Chennai','Savings',61000,'Active',695,'2023-07-22',203),
(111,'Ramesh',44,'Male','Bangalore','Current',145000,'Active',775,'2020-02-11',201),
(112,'Lakshmi',36,'Female','Mumbai','Savings',97000,'Active',740,'2021-04-19',204),
(113,'Amit',48,'Male','Delhi','Savings',52000,'Inactive',660,'2020-10-30',205),
(114,'Pooja',26,'Female','Pune','Savings',39000,'Active',705,'2024-01-08',206),
(115,'Naveen',37,'Male','Kochi','Current',132000,'Active',755,'2021-12-16',207),
(116,'Swathi',30,'Female','Hyderabad','Savings',71000,'Active',715,'2023-05-27',202),
(117,'Manoj',43,'Male','Chennai','Current',158000,'Active',785,'2020-07-13',203),
(118,'Deepika',33,'Female','Bangalore','Savings',None,'Active',730,'2022-09-21',201),
(119,'Venkat',50,'Male','Mumbai','Current',195000,'Active',810,'2019-12-03',204),
(120,'Harini',29,'Female','Delhi','Savings',47000,'Active',700,'2024-02-15',205)
]

customers_df = spark.createDataFrame(
    customers_data,
    [
        'customer_id','customer_name','age','gender','city',
        'account_type','account_balance','account_status',
        'credit_score','opening_date','branch_id'
    ]
)

In [0]:
customers_df.show()

In [0]:
transactions_data = [
(10001,101,'2025-01-05','Credit',25000,'UPI','Success'),
(10002,101,'2025-01-12','Debit',8000,'Card','Success'),
(10003,102,'2025-01-08','Credit',50000,'NEFT','Success'),
(10004,102,'2025-01-18','Debit',15000,'UPI','Success'),
(10005,103,'2025-02-02','Debit',12000,'Card','Failed'),
(10006,103,'2025-02-10','Credit',30000,'NEFT','Success'),
(10007,104,'2025-02-15','Debit',22000,'UPI','Success'),
(10008,104,'2025-02-22','Credit',45000,'IMPS','Success'),
(10009,105,'2025-03-05','Debit',75000,'NEFT','Success'),
(10010,105,'2025-03-15','Credit',100000,'NEFT','Success'),
(10011,106,'2025-03-20','Debit',9000,'UPI','Success'),
(10012,107,'2025-03-25','Debit',5000,'Card','Failed'),
(10013,108,'2025-04-01','Credit',35000,'UPI','Success'),
(10014,108,'2025-04-10','Debit',7000,'UPI','Success'),
(10015,109,'2025-04-15','Credit',120000,'NEFT','Success'),
(10016,109,'2025-04-25','Debit',45000,'Card','Success'),
(10017,110,'2025-05-02','Debit',11000,'UPI','Success'),
(10018,111,'2025-05-10','Credit',65000,'IMPS','Success'),
(10019,111,'2025-05-20','Debit',18000,'UPI','Success'),
(10020,112,'2025-05-25','Debit',25000,'Card','Failed'),
(10021,113,'2025-06-01','Credit',40000,'NEFT','Success'),
(10022,114,'2025-06-05','Debit',6000,'UPI','Success'),
(10023,115,'2025-06-12','Credit',55000,'IMPS','Success'),
(10024,115,'2025-06-20','Debit',14000,'Card','Success'),
(10025,116,'2025-07-01','Credit',28000,'UPI','Success'),
(10026,117,'2025-07-10','Debit',32000,'NEFT','Success'),
(10027,118,'2025-07-15','Credit',47000,'UPI','Success'),
(10028,118,'2025-07-25','Debit',9000,'UPI','Success'),
(10029,119,'2025-08-01','Credit',90000,'NEFT','Success'),
(10030,120,'2025-08-10','Debit',7500,'Card','Success')
]

transactions_df = spark.createDataFrame(
    transactions_data,
    [
        'transaction_id','customer_id','transaction_date',
        'transaction_type','amount','payment_mode',
        'transaction_status'
    ]
)

In [0]:
transactions_df.show()

Find customers whose account_balance is NULL.

In [0]:
from pyspark.sql.functions import col

customers_df.filter(col("account_balance").isNull()).show()

In [0]:
customers_df.write.format("delta").mode('overwrite').saveAsTable("customer_prac")

customers_df.display()

In [0]:
%sql
select customer_id,customer_name,account_balance from customer_prac
where account_balance is null

In [0]:
from pyspark.sql.functions import col, when

customers_df.withColumn("account_amount", when(col("account_balance").isNull(), 0).otherwise(col("account_balance"))).display()

In [0]:
%sql
select customer_name,
       case when account_balance is null then 0 else account_balance end as account_balance
from customer_prac

In [0]:
from pyspark.sql.functions import col, when, avg
from pyspark.sql import Window

customers_df.withColumn("avg_amount", when(col("account_balance").isNull(), avg(col("account_balance")).over(Window.partitionBy())).otherwise(col("account_balance"))).show()

In [0]:
from pyspark.sql.functions import col, row_number
from pyspark.sql import Window

window_spec = Window.partitionBy(col("city")).orderBy(col("account_balance").desc())
customers_df.withColumn("rank", row_number().over(window_spec)).where(col("rank") == 2).show()

In [0]:
%sql
with cte as(
    select customer_name,city,account_balance 
    ,row_number() over(partition by city order by account_balance desc) as rnk
    from customer_prac
)select * from cte
where rnk = 2

In [0]:
df = spark.table("customer_prac").select("city", "account_balance")
df.show()

### SCD TYPE 2

In [0]:
from pyspark.sql import functions as F

customer_data = [
    (101, "Rahul", "Chennai",   50000),
    (102, "Priya", "Hyderabad", 60000),
    (103, "Ravi",  "Bangalore", 70000)
]

customer_df = spark.createDataFrame(
    customer_data,
    ["customer_id", "name", "city", "salary"]
)

customer_df.show()

In [0]:
target_data = [
    (101, "Rahul", "Bangalore", 50000, "2025-01-01", None, 1),
    (102, "Priya", "Hyderabad", 60000, "2025-01-01", None, 1)
]

from pyspark.sql.types import StructType, StructField, StringType, LongType

target_df = spark.createDataFrame(
    target_data,
    StructType([
        StructField("customer_id", LongType(), True),
        StructField("name", StringType(), True),
        StructField("city", StringType(), True),
        StructField("salary", LongType(), True),
        StructField("start_date", StringType(), True),
        StructField("end_date", StringType(), True),
        StructField("is_current", LongType(), True)
    ])
)

target_df = target_df.withColumn(
    "start_date",
    F.to_date("start_date")
).withColumn(
    "end_date",
    F.to_date("end_date")
)

target_df.show()

In [0]:
customer_df = customer_df.withColumn(
    "source_hash",
    F.sha2(
        F.concat_ws(
            "||",
            F.coalesce(F.col("name"), F.lit("")),
            F.coalesce(F.col("city"), F.lit("")),
            F.coalesce(F.col("salary").cast("string"), F.lit(""))
        ),
        256
    )
)

customer_df.show(truncate=False)

In [0]:
target_df = target_df.withColumn(
    "target_hash",
    F.sha2(
        F.concat_ws(
            "||",
            F.coalesce(F.col("name"), F.lit("")),
            F.coalesce(F.col("city"), F.lit("")),
            F.coalesce(F.col("salary").cast("string"), F.lit(""))
        ),
        256
    )
)

target_df.show(truncate=False)

In [0]:
joined_df = (
    customer_df.alias("s")
    .join(
        target_df.alias("t"),
        F.col("s.customer_id") == F.col("t.customer_id"),
        "left"
    )
)
joined_df.select(
    F.col("s.customer_id").alias("customer_id"),
    F.col("s.name").alias("source_name"),
    F.col("s.city").alias("source_city"),
    F.col("t.name").alias("target_name"),
    F.col("t.city").alias("target_city"),
    F.col("s.source_hash"),
    F.col("t.target_hash")
).show(truncate=False)

In [0]:
comparison_df = joined_df.withColumn(
    "record_status",

    F.when(
        F.col("t.customer_id").isNull(),
        "NEW"
    )

    .when(
        F.col("s.source_hash") != F.col("t.target_hash"),
        "CHANGED"
    )

    .otherwise(
        "UNCHANGED"
    )
)

In [0]:
comparison_df.show()

In [0]:
comparison_df.select(
    F.col("s.customer_id").alias("customer_id"),
    F.col("s.city").alias("source_city"),
    F.col("t.city").alias("target_city"),
    "record_status"
).show()

In [0]:
new_records = comparison_df.filter(
    F.col("record_status") == "NEW"
)

In [0]:
new_records_selected = new_records.select(
    F.col("s.customer_id").alias("customer_id"),
    F.col("s.name").alias("name"),
    F.col("s.city").alias("city"),
    F.col("s.salary").alias("salary"),
    F.current_date().alias("start_date"),
    F.lit(None).cast("date").alias("end_date"),
    F.lit(1).alias("is_current"),
    F.col("s.source_hash").alias("record_hash")
)
new_records_selected.show()

In [0]:
changed_records = comparison_df.filter(
    F.col("record_status") == "CHANGED"
)
changed_records.select(
    F.col("s.customer_id").alias("customer_id"),
    F.col("t.city").alias("old_city"),
    F.col("s.city").alias("new_city"),
    F.col("t.salary").alias("old_salary"),
    F.col("s.salary").alias("new_salary")
).show()

In [0]:
target_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dim_customer_scd")

In [0]:
changed_ids = changed_records.select(
    F.col("s.customer_id").alias("customer_id")
)

changed_ids.createOrReplaceTempView("changed_ids")

In [0]:
%sql
MERGE INTO dim_customer_scd AS target
USING changed_ids AS source

ON target.customer_id = source.customer_id
AND target.is_current = 1

WHEN MATCHED THEN
UPDATE SET
    target.end_date = current_date(),
    target.is_current = 0

In [0]:
spark.table("dim_customer_scd").show()

In [0]:
changed_new_versions = changed_records.select(
    F.col("s.customer_id").alias("customer_id"),
    F.col("s.name").alias("name"),
    F.col("s.city").alias("city"),
    F.col("s.salary").alias("salary"),
    F.current_date().alias("start_date"),
    F.lit(None).cast("date").alias("end_date"),
    F.lit(1).alias("is_current"),
    F.col("s.source_hash").alias("record_hash")
)

In [0]:
records_to_insert = new_records_selected.unionByName(
    changed_new_versions
)


In [0]:
records_to_insert.show()

In [0]:
records_to_insert.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true")\
    .saveAsTable("dim_customer_scd")

In [0]:
spark.table("dim_customer_scd") \
    .orderBy("customer_id", "start_date") \
    .show()

### CDC Implemention

In [0]:
%sql
CREATE OR REPLACE TABLE target_data (
    customer_id INT,
    name STRING,
    city STRING,
    salary INT
)
USING DELTA;

In [0]:
%sql
INSERT INTO target_data VALUES
(101, 'Rahul', 'Bangalore', 50000),
(102, 'Priya', 'Mumbai', 60000),
(103, 'Amit', 'Delhi', 70000);

In [0]:
%sql
select * from target_data

In [0]:
%sql
CREATE OR REPLACE TABLE source_data (
    customer_id INT,
    name STRING,
    city STRING,
    salary INT
)
USING DELTA;

In [0]:
%sql
INSERT INTO source_data VALUES
(101, 'Rahul', 'Bangalore', 55000),
(102, 'Priya', 'Pune', 60000),
(104, 'Sneha', 'Chennai', 65000);

In [0]:
%sql
MERGE INTO target_data AS target
USING source_data AS source

ON target.customer_id = source.customer_id

WHEN MATCHED THEN
    UPDATE SET
        target.name = source.name,
        target.city = source.city,
        target.salary = source.salary

WHEN NOT MATCHED THEN
    INSERT (
        customer_id,
        name,
        city,
        salary
    )
    VALUES (
        source.customer_id,
        source.name,
        source.city,
        source.salary
    );